# 06 — Baselines

**Étape 8 — Baselines (moyenne, J-1, J-7)**

Construire des références naïves solides avant tout modèle sophistiqué
(section 10, niveaux 0-1). Ces baselines serviront de référence obligatoire dans le
tableau de benchmark final.


## Setup

In [4]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import data, features, models, backtest, metrics, plots

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (14, 4)


## Split chronologique train/val/test (`src/backtest.py::chronological_split`)

In [2]:
from pathlib import Path

DATA_PATH = Path("../data/processed/dataset_features.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(
    df["datetime"],
    utc=True,
)

df = (
    df.set_index("datetime")
      .sort_index()
)

assert df.index.is_unique
assert df.index.is_monotonic_increasing
assert df.shape == (32855, 42), f"Dimensions inattendues : {df.shape}"

# Les frontières sont exprimées en heure française.
# backtest.chronological_split les convertit automatiquement vers l’UTC.
TRAIN_END = pd.Timestamp(
    "2024-01-01 00:00",
    tz="Europe/Paris",
)

VALIDATION_END = pd.Timestamp(
    "2025-01-01 00:00",
    tz="Europe/Paris",
)

train, validation, test = backtest.chronological_split(
    df,
    train_end=TRAIN_END,
    val_end=VALIDATION_END,
)

print("Dataset complet :", df.shape)
print()
print("Train :", train.shape)
print(
    train.index.min(),
    "→",
    train.index.max(),
)
print()
print("Validation :", validation.shape)
print(
    validation.index.min(),
    "→",
    validation.index.max(),
)
print()
print("Test :", test.shape)
print(
    test.index.min(),
    "→",
    test.index.max(),
)

assert len(train) == 17520
assert len(validation) == 8784
assert len(test) == 6551

assert train.index.max() < validation.index.min()
assert validation.index.max() < test.index.min()

Dataset complet : (32855, 42)

Train : (17520, 42)
2021-12-31 23:00:00+00:00 → 2023-12-31 22:00:00+00:00

Validation : (8784, 42)
2023-12-31 23:00:00+00:00 → 2024-12-31 22:00:00+00:00

Test : (6551, 42)
2024-12-31 23:00:00+00:00 → 2025-09-30 21:00:00+00:00


## Baseline niveau 0 : ajuster sur train (`fit_mean_by_hour`) puis prédire (`predict_mean_by_hour`)

In [3]:
# Apprentissage uniquement sur le train
hourly_means = models.fit_mean_by_hour(
    train_df=train,
    price_col="price_da",
    hour_col="hour",
)

# Tableau commun qui contiendra la cible et les prédictions
predictions = pd.DataFrame(
    {
        "y_true": df["price_da"],
    },
    index=df.index,
)

predictions["Moyenne horaire"] = models.predict_mean_by_hour(
    df=df,
    hourly_means=hourly_means,
    hour_col="hour",
)

print("Moyennes apprises uniquement sur le train :")
display(
    hourly_means.rename("prix_moyen_train_EUR_MWh").to_frame()
)

display(predictions.head())

Moyennes apprises uniquement sur le train :


,prix_moyen_train_EUR_MWh
hour,
0,174.518219
1,161.236918
2,155.983123
3,145.878973
4,143.041411
5,153.788959
6,179.750630
7,208.328945
8,219.272370


,y_true,Moyenne horaire
datetime,,
2021-12-31 23:00:00+00:00,89.06,174.518219
2022-01-01 00:00:00+00:00,78.48,161.236918
2022-01-01 01:00:00+00:00,85.16,155.983123
2022-01-01 02:00:00+00:00,50.00,145.878973
2022-01-01 03:00:00+00:00,37.67,143.041411


## Baseline niveau 1 : Naive J-1 par date/période locale (`naive_persistence`, day_lag=1)

In [4]:
# Calcul sur le dataset complet afin que la première journée de chaque split
# puisse utiliser les prix du jour précédent.
predictions["Naive J-1"] = models.naive_persistence(
    df=df,
    price_col="price_da",
    day_lag=1,
    timezone="Europe/Paris",
)

print(
    "Valeurs disponibles pour J-1 :",
    predictions["Naive J-1"].notna().sum(),
)

print(
    "Valeurs manquantes pour J-1 :",
    predictions["Naive J-1"].isna().sum(),
)

display(
    predictions[
        ["y_true", "Naive J-1"]
    ].head(30)
)

Valeurs disponibles pour J-1 : 32824
Valeurs manquantes pour J-1 : 31


,y_true,Naive J-1
datetime,,
2021-12-31 23:00:00+00:00,89.06,NaN
2022-01-01 00:00:00+00:00,78.48,NaN
2022-01-01 01:00:00+00:00,85.16,NaN
2022-01-01 02:00:00+00:00,50.00,NaN
2022-01-01 03:00:00+00:00,37.67,NaN
2022-01-01 04:00:00+00:00,39.70,NaN
2022-01-01 05:00:00+00:00,40.59,NaN
2022-01-01 06:00:00+00:00,43.26,NaN
2022-01-01 07:00:00+00:00,49.66,NaN


## Baseline niveau 1 : Naive J-7 par date/période locale (`naive_persistence`, day_lag=7)

In [5]:
predictions["Naive J-7"] = models.naive_persistence(
    df=df,
    price_col="price_da",
    day_lag=7,
    timezone="Europe/Paris",
)

print(
    "Valeurs disponibles pour J-7 :",
    predictions["Naive J-7"].notna().sum(),
)

print(
    "Valeurs manquantes pour J-7 :",
    predictions["Naive J-7"].isna().sum(),
)

display(
    predictions[
        ["y_true", "Naive J-1", "Naive J-7"]
    ].head(180).tail(15)
)

Valeurs disponibles pour J-7 : 32680
Valeurs manquantes pour J-7 : 175


,y_true,Naive J-1,Naive J-7
datetime,,,
2022-01-07 20:00:00+00:00,213.81,229.96,NaN
2022-01-07 21:00:00+00:00,216.13,216.44,NaN
2022-01-07 22:00:00+00:00,205.74,210.00,NaN
2022-01-07 23:00:00+00:00,184.96,213.47,89.06
2022-01-08 00:00:00+00:00,162.79,189.00,78.48
2022-01-08 01:00:00+00:00,166.00,180.00,85.16
2022-01-08 02:00:00+00:00,161.63,164.90,50.00
2022-01-08 03:00:00+00:00,160.00,162.80,37.67
2022-01-08 04:00:00+00:00,154.30,165.00,39.70


## Calcul des métriques (MAE, RMSE, sMAPE) pour chaque baseline (`src/metrics.py`)

In [6]:
MODEL_COLUMNS = [
    "Moyenne horaire",
    "Naive J-1",
    "Naive J-7",
]

EVALUATION_SPLITS = {
    "validation": validation.index,
    "test": test.index,
}

metric_results = {}
evaluation_samples = {}

for split_name, split_index in EVALUATION_SPLITS.items():
    split_predictions = predictions.loc[
        split_index,
        ["y_true"] + MODEL_COLUMNS,
    ].copy()

    # Même échantillon pour les trois modèles :
    # on conserve uniquement les heures où toutes les prédictions existent.
    common_mask = split_predictions[
        ["y_true"] + MODEL_COLUMNS
    ].notna().all(axis=1)

    evaluation_df = split_predictions.loc[common_mask].copy()
    evaluation_samples[split_name] = evaluation_df

    metric_results[split_name] = {}

    for model_name in MODEL_COLUMNS:
        metric_results[split_name][model_name] = (
            metrics.summary_metrics(
                y_true=evaluation_df["y_true"],
                y_pred=evaluation_df[model_name],
            )
        )

    print(
        split_name,
        ":",
        len(evaluation_df),
        "heures évaluées sur",
        len(split_predictions),
    )

validation : 8781 heures évaluées sur 8784
test : 6549 heures évaluées sur 6551


## Premier tableau de benchmark (`summary_metrics`, `benchmark_table`)

In [7]:
benchmark_tables = []

for split_name, split_results in metric_results.items():
    split_table = (
        metrics.benchmark_table(split_results)
        .reset_index()
        .rename(columns={"index": "model"})
    )

    split_table.insert(0, "split", split_name)
    benchmark_tables.append(split_table)

baseline_benchmark = pd.concat(
    benchmark_tables,
    ignore_index=True,
)

baseline_benchmark = baseline_benchmark[
    [
        "split",
        "model",
        "MAE",
        "RMSE",
        "sMAPE",
        "Bias",
        "n_obs",
    ]
]

display(
    baseline_benchmark.style.format(
        {
            "MAE": "{:.2f}",
            "RMSE": "{:.2f}",
            "sMAPE": "{:.2f}",
            "Bias": "{:.2f}",
            "n_obs": "{:.0f}",
        }
    )
)

,split,model,MAE,RMSE,sMAPE,Bias,n_obs
0,validation,Naive J-1,20.36,28.35,61.04,0.24,8781
1,validation,Naive J-7,27.80,37.10,72.78,1.67,8781
2,validation,Moyenne horaire,128.54,134.49,113.06,-128.35,8781
3,test,Naive J-1,20.40,29.38,64.72,-0.14,6549
4,test,Naive J-7,31.35,43.31,78.94,-1.38,6549
5,test,Moyenne horaire,125.82,134.67,112.84,-125.55,6549
